# Passo 4 – Re-treino, janela de treino e candle (Fase A: só acerto)

Variáveis do Passo 3: **10 lags + 4 grupos (20 variáveis)**. RF fixa (300 árvores, prof. 6, folha 0,5%, 50% das variáveis). Alvo: retorno % do próximo candle. **Nenhum custo, nenhum dinheiro.**

Simulamos 2024 como na vida real: em cada data de re-treino a RF aprende só com os últimos N meses e prevê até o próximo re-treino.

**Grade:** candles 5, 15, 30, 60 min × janelas 3, 6, 12, 24, 36 meses e todo o histórico × re-treino mensal, trimestral e semestral (72 combinações).
O **1 min** entra com re-treino trimestral nas 6 janelas (6 combinações): com re-treino mensal seriam 72 treinos de até 600 mil linhas cada, inviável no tempo disponível.

**Regra de escolha (definida antes de rodar):** a combinação de **maior R² em 2024** entre as que têm **R² > 0 em pelo menos 3 dos 4 trimestres**. Se nenhuma tiver, a de maior R² com essa ressalva registrada.

Antes da grade, um diagnóstico: o modelo envelhece?

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os, time
import rf_zero as rz
print("lags:", rz.N_LAGS)
win_1m = rz.carregar("../../dados/WINN_M1/WIN$N_M1.csv")
os.makedirs("saidas", exist_ok=True)
bases = {m: rz.montar_base(win_1m, m) for m in [1, 5, 15, 30, 60]}
print({m: len(c) for m, (b, c) in bases.items()}, "variáveis")

## 4.1 O modelo envelhece?

Treinamos em 11 datas (início de cada trimestre de 2022 a 2024, janela de 12 meses) e, **sem re-treinar**, medimos o R² em cada um dos 6 meses seguintes (juntando as previsões das 11 datas). Se o R² cai com o tempo, re-treinar com mais frequência ajudaria.

In [ ]:
ancoras = pd.date_range("2022-01-01", "2024-07-01", freq="QS")
env = []
for m in [15, 60]:
    base, cols = bases[m]
    for a in ancoras:
        tr = base[(base.index >= a - pd.DateOffset(months=12)) & (base.index < a)]
        te = base[(base.index >= a) & (base.index < a + pd.DateOffset(months=6))]
        p = rz.rf_padrao(len(tr)).fit(tr[cols], tr["ret_prox"]).predict(te[cols])
        mes = (te.index.year - a.year) * 12 + (te.index.month - a.month) + 1
        env.append(pd.DataFrame({"candle": f"{m} min", "mes": mes, "prev": p, "real": te["ret_prox"].values}))
env = pd.concat(env)
curva = env.groupby(["candle", "mes"]).apply(lambda g: rz.r2_oos(g["prev"], g["real"])).unstack(0)
curva.to_csv("saidas/A4_envelhecimento.csv")
curva.round(3)

In [ ]:
ax = curva.plot(marker="o", figsize=(8, 3.6))
ax.axhline(0, color="gray", lw=.8); ax.set_xlabel("meses desde o treino"); ax.set_ylabel("R² (%)")
ax.set_title("R² por mês após o treino, sem re-treinar"); plt.tight_layout(); plt.savefig("passo04_envelhecimento.png", dpi=120); plt.show()

## 4.2 A grade

In [ ]:
ARQ = "saidas/A4_previsoes.pkl"
FREQS = {"mensal": "MS", "trimestral": "QS", "semestral": "6MS"}
JANELAS = [3, 6, 12, 24, 36, 120]   # 120 meses > histórico = todo o histórico
combos = [(m, j, f) for m in [5, 15, 30, 60] for j in JANELAS for f in FREQS] + [(1, j, "trimestral") for j in JANELAS]
prevs = pd.read_pickle(ARQ) if os.path.exists(ARQ) else {}
t0 = time.time()
for m, j, f in combos:
    if (m, j, f) in prevs: continue
    base, cols = bases[m]
    prevs[(m, j, f)] = rz.walk_forward(base, cols, rz.VALIDACAO, j, FREQS[f])[["prev", "ret_prox", "mov_prox", "close"]]
    pd.to_pickle(prevs, ARQ)
print(f"{len(prevs)} combinações, {time.time() - t0:.0f} s")

In [ ]:
grade = pd.DataFrame({k: rz.resumo_acerto(v) for k, v in prevs.items()}).T
grade.index.names = ["candle", "janela", "re-treino"]
grade = grade.sort_index(); grade.to_csv("saidas/A4_grade.csv")
cand = grade[grade["trimestres R²>0"] >= 3]
ESCOLHA = (cand if len(cand) else grade)["R² %"].astype(float).idxmax()
print("Candidatas (R² > 0 em ≥ 3 trimestres):", len(cand))
print("Escolha pela regra →", ESCOLHA)
grade.sort_values("R² %", ascending=False).head(12).round(3)

In [ ]:
# R² por candle × janela, para cada frequência
for f in FREQS:
    print(f"\nre-treino {f}")
    display(grade.xs(f, level="re-treino")["R² %"].astype(float).unstack("janela").round(3))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for a, f in zip(axs, FREQS):
    g = grade.xs(f, level="re-treino")["R² %"].astype(float).unstack("janela").drop(index=1, errors="ignore")
    im = a.imshow(g.values, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
    for i in range(g.shape[0]):
        for k in range(g.shape[1]):
            a.text(k, i, f"{g.values[i, k]:+.2f}", ha="center", va="center", fontsize=8)
    a.set_xticks(range(g.shape[1])); a.set_xticklabels(["tudo" if c == 120 else f"{c} m" for c in g.columns])
    a.set_yticks(range(g.shape[0])); a.set_yticklabels([f"{c} min" for c in g.index]); a.set_title(f"R² % · re-treino {f}")
plt.tight_layout(); plt.savefig("passo04_grade.png", dpi=120); plt.show()

In [ ]:
# Qual dimensão pesa mais? R² médio por nível (sem 1 min)
g = grade.drop(index=1, level="candle")
pd.concat({d: g.groupby(level=d)["R² %"].mean() for d in ["candle", "janela", "re-treino"]}).round(3)

## Conclusões do Passo 4

*(preenchidas após rodar)*